In [ ]:
%load_ext autoreload
%autoreload 2

import json
import torch
import pandas as pd
from torch.utils.data import DataLoader
from neural_models.dataset import NonogramsDataset
from neural_models.models.recursive_mlp import RecursiveMLP
from neural_models.models.recursive_grid_mlp import RecursiveGridMLP
from neural_models.criterions import AbstainLoss
from neural_models.train import Trainer
import itertools
from utils.display import display_grid, display_blended, display_blended_greyscale
from matplotlib import pyplot as plt

import os
if "original_dir" not in globals():
    original_dir = os.getcwd()

os.chdir(os.path.join(original_dir, "../../.."))
os.environ["DATA_DIR"] = os.path.join(os.getcwd(), "data")
os.environ["MODEL_DIR"] = os.path.join(os.getcwd(), "checkpoints")

In [ ]:
with open("data/traces_dataset/nonograms/test.jsonl", "r") as f:
    json_dataset = [json.loads(line) for line in f.readlines()]

In [ ]:
traces = [
    {"puzzle_idx": i, "method": m, **run}
    for i, p in enumerate(json_dataset)
    for m, d in p["traces"].items()
    for run in next(iter(d.values()))   # unwraps {'1.0': [run, ...]}
]

df = pd.json_normalize(traces)

In [ ]:
df

In [ ]:
df.groupby(["method", "stopping_reason"])["num_steps"].mean()

In [ ]:
import matplotlib.pyplot as plt

# Crosstab: rows = method, cols = stopping_reason, values = fraction
pct = pd.crosstab(df["method"], df["stopping_reason"], normalize="index")

# Optional: fixed column order + nice colors
pct = pct.reindex(columns=["SOLVED", "NO_CHANGE", "TIMEOUT"], fill_value=0)
colors = {"SOLVED": "#4caf50", "NO_CHANGE": "#ff9800", "TIMEOUT": "#e53935"}

ax = pct.plot(kind="barh", stacked=True,
              color=[colors[c] for c in pct.columns],
              figsize=(8, 0.5 * len(pct) + 2),
              width=0.8)

ax.set_xlabel("Fraction of runs")
ax.set_ylabel("")
ax.set_xlim(0, 1)
ax.legend(title="stopping_reason", bbox_to_anchor=(1.02, 1), loc="upper left")

# Annotate percentages inside each segment
for i, method in enumerate(pct.index):
    left = 0
    for reason in pct.columns:
        val = pct.loc[method, reason]
        if val > 0.03:  # skip tiny labels
            ax.text(left + val / 2, i, f"{val:.0%}",
                    ha="center", va="center", color="white", fontsize=9)
        left += val

plt.tight_layout()
plt.show()

In [ ]:
from utils.display import display_grid, display_blended, display_blended_greyscale
import numpy as np
example = df.loc[df["stopping_reason"] == "SOLVED", "steps"].iloc[0]
clues = json_dataset[df.loc[df["stopping_reason"] == "SOLVED", "puzzle_idx"].iloc[0]]["clues"]
max_width = 4
figs, axes = plt.subplots(len(example) // max_width + (len(example) % max_width > 0), max_width,  figsize=(max_width * 3, (len(example) // max_width + (len(example) % max_width > 0)) * 3))
for i, step in enumerate(example):
    display_grid(axes.flat[i], np.array(step), clues=clues, title=f"Step {i+1}")